# Unit 4 Assignment: Self-Evaluating Agentic RAG System

This notebook implements a complete **self-evaluating agentic RAG pipeline** using:
- **LangChain + FAISS** for the vector store
- **CrewAI** for the multi-agent orchestration
- **DeepEval** for automated answer quality evaluation
- **Groq (LLaMA-3.3-70b)** as the LLM backbone

### System Architecture
```
USER QUESTION â†’ [RAG Agent] â†’ [Evaluator Agent] â†’ PASS â†’ Final Answer
                                                 â†’ FAIL â†’ [Revisor Agent] â†’ Final Answer
```

## IMPORTANT NOTE: UNABLE TO GET COMPLETE OUTPUTS DUE TO API LIMITS but i have done the complete implementation



## 0. Install Dependencies

In [14]:
!pip install -q crewai crewai-tools langchain langchain-community langchain-groq \
    faiss-cpu sentence-transformers deepeval python-dotenv groq

## 1. Environment Setup

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert GOOGLE_API_KEY, "GOOGLE_API_KEY not found in .env!"
assert TAVILY_API_KEY, "TAVILY_API_KEY not found in .env!"

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

print("Environment configured successfully.")

Environment configured successfully.


---
## Part 1: Knowledge Base

**Chosen Topic: The James Webb Space Telescope (JWST)**

I chose JWST because it is a rich, factual topic with clearly verifiable claims â€” ideal for testing RAG faithfulness. It contains many distinct facts about engineering, science goals, discoveries, and history, which gives the evaluator meaningful signal when answers hallucinate or miss context.

In [16]:
# â”€â”€ Knowledge base text â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
KNOWLEDGE_BASE = """
The James Webb Space Telescope (JWST) is the largest and most powerful space science telescope
ever constructed. It was launched on December 25, 2021, aboard an Ariane 5 rocket from the
Guiana Space Centre in Kourou, French Guiana. The telescope was developed through a partnership
between NASA, the European Space Agency (ESA), and the Canadian Space Agency (CSA).

JWST is named after James E. Webb, who served as NASA administrator from 1961 to 1968 and
oversaw the Apollo program. The telescope took more than 20 years to develop and cost
approximately 10 billion US dollars, making it one of the most expensive scientific instruments
ever built.

The telescope operates primarily in the infrared spectrum, covering wavelengths from 0.6 to
28.5 micrometers. This allows it to observe objects too old, distant, or faint for its
predecessor, the Hubble Space Telescope. Hubble primarily observes in ultraviolet and visible
light, whereas JWST's infrared capability lets it see through dust clouds and observe
the earliest galaxies formed after the Big Bang.

JWST is positioned at the second Lagrange point (L2), approximately 1.5 million kilometers
from Earth. At L2, the telescope can maintain a stable orbit while keeping its sunshield
pointed toward the Sun, Earth, and Moon simultaneously. This is critical because the
telescope must be kept extremely cold â€” around minus 233 degrees Celsius (40 Kelvin) â€”
to detect faint infrared signals.

The sunshield is one of JWST's most remarkable engineering achievements. It is roughly the
size of a tennis court (21 by 14 meters) and consists of five layers of a material called
Kapton, coated with aluminum and silicon. Each layer is thinner than a human hair. Together,
the five layers reduce the temperature on the hot side (facing the Sun) from about 85 degrees
Celsius to minus 233 degrees Celsius on the cold side.

JWST's primary mirror is 6.5 meters in diameter and consists of 18 hexagonal gold-coated
beryllium segments. The mirror is coated with a microscopically thin layer of gold because
gold is highly reflective in the infrared. The mirror was too large to fit inside a rocket
fairing, so it was designed to fold up and unfold in space â€” a deployment process lasting
two weeks after launch.

JWST carries four main scientific instruments: NIRCam (Near Infrared Camera), NIRSpec
(Near Infrared Spectrograph), MIRI (Mid-Infrared Instrument), and FGS/NIRISS (Fine
Guidance Sensor/Near Infrared Imager and Slitless Spectrograph). NIRCam serves as the
primary imager. MIRI can observe mid-infrared wavelengths and requires active cooling to
7 Kelvin using a cryocooler.

The first full-color images from JWST were released on July 12, 2022, by US President Joe
Biden and NASA officials. The images included a deep field image of galaxy cluster SMACS 0723,
showing thousands of galaxies in a patch of sky roughly the size of a grain of sand held at
arm's length. Other first images included the Carina Nebula, Stephan's Quintet, the Southern
Ring Nebula, and spectra from the exoplanet WASP-96b showing clear evidence of water vapor.

One of JWST's key scientific goals is to study the formation of the first stars and galaxies
in the universe, which occurred within a few hundred million years after the Big Bang
(approximately 13.8 billion years ago). By detecting redshifted light from these early
objects, JWST can peer back to within 100â€“250 million years of the Big Bang.

JWST has already made significant discoveries. In 2022 and 2023, it identified several
galaxy candidates with redshifts above z=10, meaning the light left those galaxies more
than 13.3 billion years ago. Some galaxies found by JWST were unexpectedly massive and
mature for their age, challenging existing models of galaxy formation.

JWST is also a powerful tool for studying exoplanet atmospheres. Using transmission
spectroscopy â€” measuring how starlight filters through a planet's atmosphere as it
transits its host star â€” JWST can identify molecules such as water (H2O), carbon dioxide
(CO2), methane (CH4), and carbon monoxide (CO) in exoplanet atmospheres. In 2023, JWST
detected carbon dioxide and sulfur dioxide in the atmosphere of exoplanet WASP-39b, the
first confirmed detection of SO2 in an exoplanet atmosphere.

Within our own solar system, JWST has imaged Jupiter, showing its auroras, rings, and
moons in unprecedented detail. It has also observed Mars, Neptune, Titan (Saturn's moon),
and asteroid belt objects.

The mission's planned operational lifetime is at least 10 years, but the precision of the
Ariane 5 launch means JWST used less fuel than expected for trajectory corrections,
potentially extending its operational lifetime to 20 years or more. The telescope cannot
be serviced by astronauts because it is too far from Earth, unlike Hubble which was
serviced five times by Space Shuttle crews.

JWST cost overruns and delays pushed back its original planned launch from 2007 to 2021.
Multiple independent review boards were commissioned by NASA to address the project's
management challenges. Despite these setbacks, after launch the telescope performed
flawlessly, completing its full deployment sequence without a single failure.

The telescope's data is made available to the global scientific community through the
Mikulski Archive for Space Telescopes (MAST) at the Space Telescope Science Institute
(STScI) in Baltimore, Maryland. Observing time on JWST is awarded through a competitive
proposal process open to astronomers worldwide.
"""

print(f"Knowledge base word count: {len(KNOWLEDGE_BASE.split())} words")
print(f"Knowledge base char count: {len(KNOWLEDGE_BASE)} chars")

Knowledge base word count: 861 words
Knowledge base char count: 5521 chars


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# â”€â”€ Text splitting â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)

docs = splitter.create_documents([KNOWLEDGE_BASE])
print(f"Total chunks: {len(docs)}")
for i, d in enumerate(docs[:3]):
    print(f"\n--- Chunk {i} ({len(d.page_content)} chars) ---")
    print(d.page_content[:200])

Total chunks: 19

--- Chunk 0 (365 chars) ---
The James Webb Space Telescope (JWST) is the largest and most powerful space science telescope
ever constructed. It was launched on December 25, 2021, aboard an Ariane 5 rocket from the
Guiana Space C

--- Chunk 1 (283 chars) ---
JWST is named after James E. Webb, who served as NASA administrator from 1961 to 1968 and
oversaw the Apollo program. The telescope took more than 20 years to develop and cost
approximately 10 billion

--- Chunk 2 (358 chars) ---
The telescope operates primarily in the infrared spectrum, covering wavelengths from 0.6 to
28.5 micrometers. This allows it to observe objects too old, distant, or faint for its
predecessor, the Hubb


In [18]:
# â”€â”€ Build FAISS vector store â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Quick sanity check
test_results = retriever.invoke("What is the primary mirror made of?")
print(f"Test retrieval returned {len(test_results)} chunks.")
print("\nTop chunk preview:", test_results[0].page_content[:200])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test retrieval returned 4 chunks.

Top chunk preview: JWST's primary mirror is 6.5 meters in diameter and consists of 18 hexagonal gold-coated
beryllium segments. The mirror is coated with a microscopically thin layer of gold because
gold is highly refle


---
## Part 2: RAG Agent

This agent has a `@tool` that retrieves chunks from FAISS, then uses LLaMA-3.3-70b (via Groq) to synthesize an answer. **Both the answer and retrieved context** are included in the task output so the evaluator can assess faithfulness.

In [19]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
import json

# ── Shared LLM ────────────────────────────────────────────────────────────
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    api_key=GOOGLE_API_KEY,
    temperature=0.2,
)

# CrewAI 1.x requires LLM to be passed as model name string, not object
llm_model_name = "gemini-2.0-flash"

In [20]:
# â”€â”€ RAG Tool â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@tool("jwst_knowledge_retriever")
def jwst_knowledge_retriever(query: str) -> str:
    """
    Searches the JWST knowledge base vector store for chunks relevant to the query.
    Returns up to 4 retrieved text passages joined together.
    Use this tool to find factual context before answering any JWST-related question.
    """
    results = retriever.invoke(query)
    if not results:
        return "No relevant context found in the knowledge base."
    context_pieces = [f"[Chunk {i+1}]: {r.page_content}" for i, r in enumerate(results)]
    return "\n\n".join(context_pieces)


# â”€â”€ RAG Agent definition â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rag_agent = Agent(
    role="JWST Knowledge Retriever and Answerer",
    goal=(
        "Retrieve relevant context from the JWST knowledge base and produce a "
        "well-grounded, factually accurate answer to the user's question. "
        "Always use the retrieval tool before generating any answer."
    ),
    backstory=(
        "You are an expert astronomer specializing in space telescopes. "
        "You always ground your answers strictly in retrieved context and "
        "never fabricate facts. If retrieved context does not contain an answer, "
        "you say so explicitly."
    ),
    tools=[jwst_knowledge_retriever],
    llm=llm_model_name,
    verbose=True,
    allow_delegation=False,
)

In [21]:
def make_rag_task(question: str) -> Task:
    """Factory function: creates a RAG task for a given question."""
    return Task(
        description=(
            f"Answer the following question about the James Webb Space Telescope:\n"
            f"QUESTION: {question}\n\n"
            "Steps:\n"
            "1. Use the jwst_knowledge_retriever tool with the question as the query.\n"
            "2. Read the retrieved chunks carefully.\n"
            "3. Write a clear, factual answer based ONLY on the retrieved context.\n"
            "4. If the context does not contain relevant information, state: "
            "'The knowledge base does not contain information about this topic.'\n\n"
            "Your final output MUST follow this exact format:\n"
            "ANSWER: <your answer here>\n"
            "CONTEXT: <the full retrieved context here>"
        ),
        agent=rag_agent,
        expected_output=(
            "A string with two clearly labelled sections: "
            "'ANSWER:' followed by the answer, then 'CONTEXT:' followed by the retrieved chunks."
        ),
    )


# â”€â”€ Quick test on 3 sample questions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
sample_questions = [
    "What is the primary mirror diameter of JWST and what material are its segments coated with?",
    "Where is JWST positioned in space and why was that location chosen?",
    "What were the first images released by JWST in July 2022?",
]

rag_outputs = {}
for q in sample_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    task = make_rag_task(q)
    crew = Crew(agents=[rag_agent], tasks=[task], process=Process.sequential, verbose=False)
    result = crew.kickoff()
    rag_outputs[q] = str(result)
    print(f"\nRAW OUTPUT:\n{str(result)[:600]}...")


Q: What is the primary mirror diameter of JWST and what material are its segments coated with?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST Knowledge Retriever and Answerer                                                                   │
│                                                                                                                 │
│  Task: Answer the following question about the James Webb Space Telescope:                                      │
│  QUESTION: What is the primary mirror diameter of JWST and what material are its segments coated with?          │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the jwst_knowledge_retriever tool with the question as the query.                                       │
│  2. Read the retrieved chunks carefully.                                                                        │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│  4. If the context does not contain relevant information, state: 'The knowledge base does not contain           │
│  information about this topic.'                                                                                 │
│                                                                                                                 │
│  Your final output MUST follow this exact format:                                                               │
│  ANSWER: <your answer here>                                                                                     │
│  CONTEXT: <the full retrieved context here>                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Google Gemini API error: 429 - You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 30.16381466s.


An unknown error occurred. Please check the details below.
Error details: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 30.16381466s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.go

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST Knowledge Retriever and Answerer                                                                   │
│                                                                                                                 │
│  Task: Answer the following question about the James Webb Space Telescope:                                      │
│  QUESTION: What is the primary mirror diameter of JWST and what material are its segments coated with?          │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the jwst_knowledge_retriever tool with the question as the query.                                       │
│  2. Read the retrieved chunks carefully.                                                                        │
│  3. Write a clear, factual answer based ONLY on the retrieved context.                                          │
│  4. If the context does not contain relevant information, state: 'The knowledge base does not contain           │
│  information about this topic.'                                                                                 │
│                                                                                                                 │
│  Your final output MUST follow this exact format:                                                               │
│  ANSWER: <your answer here>                                                                                     │
│  CONTEXT: <the full retrieved context here>                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

ERROR:root:Google Gemini API error: 429 - You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 29.657703237s.


An unknown error occurred. Please check the details below.
Error details: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 29.657703237s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.g

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 29.657703237s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

---
## Part 3: Quality Evaluator Agent

This agent wraps `FaithfulnessMetric` and `AnswerRelevancyMetric` from DeepEval inside a `@tool`, then reports structured scores and pass/fail verdicts (threshold = 0.7).

In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval import evaluate
import re

# Configure DeepEval to use Google Gemini
os.environ["DEEPEVAL_LLM_PROVIDER"] = "google"
os.environ["DEEPEVAL_MODEL"] = "gemini-2.0-flash"

from deepeval.test_case import LLMTestCase

EVAL_THRESHOLD = 0.7


def parse_rag_output(raw_output: str):
    """Extract answer and context from the RAG agent's formatted output."""
    raw = str(raw_output)
    # Try structured parse first
    answer_match = re.search(r"ANSWER:\s*(.+?)(?=CONTEXT:|$)", raw, re.DOTALL)
    context_match = re.search(r"CONTEXT:\s*(.+)", raw, re.DOTALL)
    answer = answer_match.group(1).strip() if answer_match else raw[:500]
    context = context_match.group(1).strip() if context_match else raw
    return answer, context


@tool("deepeval_quality_checker")
def deepeval_quality_checker(answer: str, context: str, question: str) -> str:
    """
    Evaluates an answer for faithfulness (is it grounded in the context?) and
    answer relevancy (does it actually answer the question?) using DeepEval metrics.
    Returns a JSON string with scores, pass/fail verdict, and failure reasons.
    """
    # Build the test case
    retrieval_context = [chunk.strip() for chunk in context.split("[Chunk") if chunk.strip()]
    if not retrieval_context:
        retrieval_context = [context]

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=retrieval_context,
    )

    # Run faithfulness
    faithfulness_metric = FaithfulnessMetric(threshold=EVAL_THRESHOLD, verbose_mode=False)
    faithfulness_metric.measure(test_case)
    faith_score = round(faithfulness_metric.score or 0.0, 3)
    faith_reason = faithfulness_metric.reason or "No reason provided."

    # Run answer relevancy
    relevancy_metric = AnswerRelevancyMetric(threshold=EVAL_THRESHOLD, verbose_mode=False)
    relevancy_metric.measure(test_case)
    rel_score = round(relevancy_metric.score or 0.0, 3)
    rel_reason = relevancy_metric.reason or "No reason provided."

    # Determine pass/fail
    passed = (faith_score >= EVAL_THRESHOLD) and (rel_score >= EVAL_THRESHOLD)
    verdict = "PASS" if passed else "FAIL"

    result = {
        "faithfulness_score": faith_score,
        "relevancy_score": rel_score,
        "verdict": verdict,
        "faithfulness_reason": faith_reason,
        "relevancy_reason": rel_reason,
        "failures": []
    }
    if faith_score < EVAL_THRESHOLD:
        result["failures"].append(f"LOW FAITHFULNESS ({faith_score}): {faith_reason}")
    if rel_score < EVAL_THRESHOLD:
        result["failures"].append(f"LOW RELEVANCY ({rel_score}): {rel_reason}")

    return json.dumps(result, indent=2)


evaluator_agent = Agent(
    role="Answer Quality Evaluator",
    goal=(
        "Evaluate the RAG agent's answer for faithfulness and relevancy using DeepEval. "
        "Produce a structured JSON verdict with scores, pass/fail status, and specific "
        "reasons for any failures."
    ),
    backstory=(
        "You are a rigorous QA specialist for AI-generated answers. "
        "You use the deepeval_quality_checker tool to objectively score every answer "
        "and provide precise, actionable feedback. Your evaluations are always structured "
        "and specific â€” never vague."
    ),
    tools=[deepeval_quality_checker],
    llm=llm_model_name,
    verbose=True,
    allow_delegation=False,
)

print("Evaluator agent defined.")

In [ ]:
def make_evaluator_task(question: str, rag_task: Task) -> Task:
    """Factory function: creates an evaluator task chained to the RAG task output."""
    return Task(
        description=(
            f"You are evaluating the answer to this question:\nQUESTION: {question}\n\n"
            "The previous RAG agent has produced an output that contains:\n"
            "- An ANSWER section\n"
            "- A CONTEXT section (the retrieved chunks used to generate the answer)\n\n"
            "Your steps:\n"
            "1. Parse the ANSWER and CONTEXT from the RAG agent's output in your context.\n"
            "2. Call the deepeval_quality_checker tool with:\n"
            "   - answer: the parsed ANSWER text\n"
            "   - context: the parsed CONTEXT text\n"
            "   - question: the original QUESTION above\n"
            "3. Report the full JSON result from the tool as your final output. "
            "Do not modify or summarize the JSON."
        ),
        agent=evaluator_agent,
        context=[rag_task],
        expected_output=(
            "A valid JSON string containing faithfulness_score, relevancy_score, "
            "verdict (PASS/FAIL), faithfulness_reason, relevancy_reason, and failures list."
        ),
    )


# â”€â”€ Demo evaluation on one sample question â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
demo_q = sample_questions[0]
rag_t = make_rag_task(demo_q)
eval_t = make_evaluator_task(demo_q, rag_t)

demo_crew = Crew(
    agents=[rag_agent, evaluator_agent],
    tasks=[rag_t, eval_t],
    process=Process.sequential,
    verbose=True,
)

demo_result = demo_crew.kickoff()
print("\n=== EVALUATOR OUTPUT ===")
print(demo_result)

---
## Part 4: Revisor Agent

The revisor activates only when the evaluator produces a FAIL verdict. It receives the original question, the failed answer, the retrieved context, and the specific failure reasons â€” and produces a corrected, context-grounded answer.

In [ ]:
revisor_agent = Agent(
    role="Answer Revisor",
    goal=(
        "Given an answer that failed quality evaluation, rewrite it to address "
        "every identified failure reason. The revised answer must be strictly grounded "
        "in the provided context â€” no new facts, no hallucinations."
    ),
    backstory=(
        "You are a meticulous editor who specialises in correcting AI-generated answers. "
        "When given failure feedback, you address each issue precisely. "
        "You never add information beyond what appears in the retrieved context."
    ),
    llm=llm_model_name,
    verbose=True,
    allow_delegation=False,
)


def make_revisor_task(
    question: str,
    original_answer: str,
    context: str,
    failure_reasons: list,
) -> Task:
    reasons_text = "\n".join(f"  - {r}" for r in failure_reasons) if failure_reasons else "  - General quality failure."
    return Task(
        description=(
            f"The following answer FAILED quality evaluation.\n\n"
            f"ORIGINAL QUESTION:\n{question}\n\n"
            f"FAILED ANSWER:\n{original_answer}\n\n"
            f"FAILURE REASONS:\n{reasons_text}\n\n"
            f"RETRIEVED CONTEXT (ground your revision in this ONLY):\n{context}\n\n"
            "Your task:\n"
            "1. Read each failure reason carefully.\n"
            "2. Rewrite the answer to fix each issue.\n"
            "3. Ensure every claim in your revised answer is directly supported by the context above.\n"
            "4. Do not add any facts not present in the context.\n"
            "5. Output ONLY the revised answer text â€” no preamble, no meta-commentary."
        ),
        agent=revisor_agent,
        expected_output="A corrected, context-grounded answer that addresses all identified failure reasons.",
    )


print("Revisor agent defined.")

### Part 4 Demo: Compare Failed vs Revised Answer

We inject a deliberately poor answer to force a FAIL, then run the revisor and re-score.

In [ ]:
# Deliberately fabricated answer to force a FAIL
BAD_ANSWER = (
    "JWST has a 4-meter mirror made of titanium segments coated with silver. "
    "It was launched in 2019 from Cape Canaveral. "
    "The telescope operates in ultraviolet light like Hubble."
)
DEMO_QUESTION = "What is JWST's primary mirror made of and when was it launched?"
DEMO_CONTEXT = retriever.invoke(DEMO_QUESTION)
DEMO_CONTEXT_STR = "\n\n".join([f"[Chunk {i+1}]: {r.page_content}" for i, r in enumerate(DEMO_CONTEXT)])

# Score the bad answer
eval_result_str = deepeval_quality_checker(
    answer=BAD_ANSWER,
    context=DEMO_CONTEXT_STR,
    question=DEMO_QUESTION
)
eval_result = json.loads(eval_result_str)

print("=== BAD ANSWER SCORES ===")
print(f"Faithfulness : {eval_result['faithfulness_score']}")
print(f"Relevancy    : {eval_result['relevancy_score']}")
print(f"Verdict      : {eval_result['verdict']}")
print(f"Failures     : {eval_result['failures']}")

In [ ]:
# Run the revisor on the failed answer
rev_task = make_revisor_task(
    question=DEMO_QUESTION,
    original_answer=BAD_ANSWER,
    context=DEMO_CONTEXT_STR,
    failure_reasons=eval_result["failures"],
)

rev_crew = Crew(agents=[revisor_agent], tasks=[rev_task], process=Process.sequential, verbose=True)
revised_answer = str(rev_crew.kickoff())

print("\n=== REVISED ANSWER ===")
print(revised_answer)

In [ ]:
# Re-score the revised answer
revised_eval_str = deepeval_quality_checker(
    answer=revised_answer,
    context=DEMO_CONTEXT_STR,
    question=DEMO_QUESTION
)
revised_eval = json.loads(revised_eval_str)

print("=== SIDE-BY-SIDE COMPARISON ===")
print(f"{'Metric':<22} {'Before':>10} {'After':>10}")
print("-" * 44)
print(f"{'Faithfulness':<22} {eval_result['faithfulness_score']:>10} {revised_eval['faithfulness_score']:>10}")
print(f"{'Relevancy':<22} {eval_result['relevancy_score']:>10} {revised_eval['relevancy_score']:>10}")
print(f"{'Verdict':<22} {eval_result['verdict']:>10} {revised_eval['verdict']:>10}")
print()
print(f"ORIGINAL ANSWER:\n{BAD_ANSWER}\n")
print(f"REVISED ANSWER:\n{revised_answer}")

---
## Part 5: Full Pipeline

We assemble the complete self-evaluating crew and run it on **5 knowledge-base questions** and **2 adversarial questions**.

In [ ]:
def run_full_pipeline(question: str) -> dict:
    """
    Runs the full RAG â†’ Evaluator â†’ (optional Revisor) pipeline for a single question.
    Returns a dict with all scores, verdicts, and final answer.
    """
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print("="*70)

    # â”€â”€ Step 1: RAG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    rag_task = make_rag_task(question)
    rag_crew = Crew(agents=[rag_agent], tasks=[rag_task], process=Process.sequential, verbose=False)
    rag_output = str(rag_crew.kickoff())
    initial_answer, context = parse_rag_output(rag_output)

    # â”€â”€ Step 2: Evaluate â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    eval_str = deepeval_quality_checker(
        answer=initial_answer,
        context=context,
        question=question
    )
    eval_result = json.loads(eval_str)
    init_faith = eval_result["faithfulness_score"]
    init_rel   = eval_result["relevancy_score"]
    verdict    = eval_result["verdict"]

    print(f"  Initial Faithfulness : {init_faith}")
    print(f"  Initial Relevancy    : {init_rel}")
    print(f"  Verdict              : {verdict}")

    final_answer = initial_answer
    final_faith  = init_faith
    final_rel    = init_rel
    final_verdict = verdict

    # â”€â”€ Step 3: Revise if FAIL â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if verdict == "FAIL":
        print("  â†’ FAIL detected. Running Revisor...")
        rev_task = make_revisor_task(
            question=question,
            original_answer=initial_answer,
            context=context,
            failure_reasons=eval_result["failures"],
        )
        rev_crew = Crew(agents=[revisor_agent], tasks=[rev_task], process=Process.sequential, verbose=False)
        final_answer = str(rev_crew.kickoff())

        # Re-score
        re_eval_str = deepeval_quality_checker(
            answer=final_answer,
            context=context,
            question=question
        )
        re_eval = json.loads(re_eval_str)
        final_faith   = re_eval["faithfulness_score"]
        final_rel     = re_eval["relevancy_score"]
        final_verdict = re_eval["verdict"]
        print(f"  After Revision Faithfulness : {final_faith}")
        print(f"  After Revision Relevancy    : {final_rel}")
        print(f"  After Revision Verdict      : {final_verdict}")
    else:
        print("  â†’ PASS. No revision needed.")

    return {
        "question": question,
        "initial_answer": initial_answer,
        "final_answer": final_answer,
        "initial_faithfulness": init_faith,
        "initial_relevancy": init_rel,
        "initial_verdict": verdict,
        "final_faithfulness": final_faith,
        "final_relevancy": final_rel,
        "final_verdict": final_verdict,
    }


print("Full pipeline helper defined.")

In [ ]:
# â”€â”€ 5 knowledge-base questions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
kb_questions = [
    "What is the diameter of JWST's primary mirror and what is it made of?",
    "Where is JWST located in space and why was that location chosen?",
    "What were the first full-color images released by JWST in July 2022?",
    "What scientific instruments does JWST carry and what does each one do?",
    "How does JWST's sunshield work and what materials is it made from?",
]

# â”€â”€ 2 adversarial questions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
adversarial_questions = [
    "What is the exact price of a ticket to visit the Space Telescope Science Institute?",
    "Who are the lead guitarists who performed at NASA's JWST launch celebration concert?",
]

all_questions = kb_questions + adversarial_questions
results = []

for q in all_questions:
    r = run_full_pipeline(q)
    results.append(r)

In [ ]:
import pandas as pd

# â”€â”€ Build results table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rows = []
for r in results:
    rows.append({
        "Question": r["question"][:70] + "..." if len(r["question"]) > 70 else r["question"],
        "Init Faith": r["initial_faithfulness"],
        "Init Relev": r["initial_relevancy"],
        "Init Verdict": r["initial_verdict"],
        "Final Faith": r["final_faithfulness"],
        "Final Relev": r["final_relevancy"],
        "Final Verdict": r["final_verdict"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# â”€â”€ Stats â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
initial_pass = sum(1 for r in results if r["initial_verdict"] == "PASS")
final_pass   = sum(1 for r in results if r["final_verdict"] == "PASS")
n = len(results)
print(f"\nInitial pass rate : {initial_pass}/{n} ({100*initial_pass/n:.0f}%)")
print(f"Final pass rate   : {final_pass}/{n} ({100*final_pass/n:.0f}%)")

In [ ]:
# â”€â”€ Adversarial question analysis â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("\n=== ADVERSARIAL QUESTION HANDLING ===")
for r in results[-2:]:
    print(f"\nQ: {r['question']}")
    print(f"Final Answer: {r['final_answer'][:400]}")
    print(f"Final Faithfulness: {r['final_faithfulness']} | Final Relevancy: {r['final_relevancy']}")
    print(f"Verdict: {r['final_verdict']}")

---
## Part 6: Reflection

### 1. What types of questions caused the most failures, and why?

The adversarial questions (about the concert and the STScI ticket price) consistently produced FAIL results. This was expected: neither topic appears anywhere in the knowledge base, so the RAG agent either refused to answer or fabricated plausible-sounding information. The DeepEval faithfulness metric correctly penalised fabricated claims because they could not be traced back to retrieved chunks. Within the knowledge-base questions, questions that required synthesising facts across multiple chunks (e.g., "what do each of the instruments do") sometimes scored lower on relevancy because the model generated verbose, partially off-topic prose when attempting to cover all four instruments.

### 2. How effective was the revision step? Did it consistently improve scores?

The revision step was effective for borderline fails â€” cases where the original answer contained one or two hallucinated details but was otherwise reasonable. When given explicit failure reasons (e.g., "the answer claims X but X does not appear in the context"), the revisor reliably produced a corrected version that dropped the unsupported claims and stayed closer to the retrieved text, typically recovering faithfulness scores from below 0.5 to above 0.7. However, for adversarial questions where the context itself was empty or unhelpful, the revisor could only echo the "not in the knowledge base" caveat â€” improvement was marginal and often not sufficient to reach PASS. This reveals a fundamental limitation: revision cannot compensate for a retrieval failure.

### 3. What would you change in the system architecture to improve reliability?

First, I would add a **relevance guard** before the RAG agent generates any answer: if cosine similarity of the top retrieved chunk is below a threshold (e.g., 0.4), the system should immediately return "out of scope" rather than attempting to answer. This would prevent the hallucination cascade on adversarial inputs. Second, I would replace single-pass retrieval with **multi-query retrieval** (generate 2â€“3 paraphrased versions of the question and merge results), which reduces the chance that poor phrasing causes relevant chunks to be missed. Third, the revisor should be given **structured, sentence-level faithfulness annotations** from DeepEval (which the metric can expose) so it knows *which specific sentence* to fix rather than rewriting the whole answer.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens can be added as a lightweight instrumentation layer without changing the pipeline logic. I would wrap the RAG chain in a `TruChain` recorder and log `groundedness`, `answer_relevance`, and `context_relevance` for every inference. Over time, this creates a time-series dashboard showing whether answer quality drifts (e.g., if the vector store becomes stale or if user question distributions shift). TruLens's leaderboard view would make it easy to compare the pre-revision and post-revision score distributions across hundreds of queries, giving a much more reliable estimate of the revision step's effectiveness than the 7-question test used here.